In [1]:
import pandas as pd

In [50]:
nodes = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='KC_Nodes')
edges = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='KC_Edges')
obs = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')

## Approach 1 : Collapse chains

First, we try grouping the KCs together to the top level :

In [21]:
import pandas as pd
import networkx as nx

G = nx.DiGraph()
G.add_nodes_from(nodes["kc_id"])

# Only use prerequisite edges to define hierarchy
G.add_edges_from(zip(edges["source_kc_id"], edges["target_kc_id"]))

# Find root nodes (no predecessors = top of the chain)
roots = [n for n in G.nodes if G.in_degree(n) == 0]
roots
# Assign each KC to its earliest ancestor (root)
kc_to_group = {}
for root in roots:
    descendants = nx.descendants(G, root) | {root}
    for kc in descendants:
        kc_to_group[kc] = root  # or overwrite with most recent ancestor

nodes["coarse_kc"] = nodes["kc_id"].map(kc_to_group)

In [54]:
nodes

,kc_id,intro_class,kc_label,description,reporting_group,ap_skill_refs,difficulty_prior,mastery_observable,modeling_note
0,KC.U1.01.statistical_context,1,Statistical context and investigative question,"Recognize the study context, what is being inv...",Unit 1: Data foundations,1.A;2.A,0.25000,Item part can be scored for this KC using MCQ ...,Use as a latent skill/KC; official AP skills a...
1,KC.U1.02.observational_unit_variable,1,Observational unit and variable,Identify what object/person is being measured ...,Unit 1: Data foundations,2.A,0.30000,Item part can be scored for this KC using MCQ ...,Use as a latent skill/KC; official AP skills a...
2,KC.U1.03.variable_type_cat_quant,1,Categorical vs quantitative variable,Classify variables as categorical or quantitat...,Unit 1: Data foundations,2.A,0.35000,Item part can be scored for this KC using MCQ ...,Use as a latent skill/KC; official AP skills a...
3,KC.U1.04.variable_type_discrete_continuous,1,Discrete vs continuous quantitative variable,Distinguish discrete count variables from cont...,Unit 1: Data foundations,2.A,0.45000,Item part can be scored for this KC using MCQ ...,Use as a latent skill/KC; official AP skills a...
4,KC.U1.05.categorical_freq_relative,1,Frequency and relative-frequency tables,"Read counts, percentages, totals, and unsuppor...",Unit 1: Categorical data,3.A;4.A,0.40000,Item part can be scored for this KC using MCQ ...,Use as a latent skill/KC; official AP skills a...
...,...,...,...,...,...,...,...,...,...
251,KC.U10.14.power_beta_relationship_extension,27,Power and beta relationship,Use and interpret the relationship power = 1 -...,Unit 10: Bootstrapping and Power,3.C|4.F,0.68000,True,Extended version of Unit 6 power/beta relation...
252,KC.U10.15.rejection_region_power_calculation,27,Power calculation using rejection region,Use a rejection region under the null and an a...,Unit 10: Bootstrapping and Power,3.C|3.E|4.F,0.82000,True,Higher-difficulty power computation KC.
253,KC.U10.16.factors_affecting_power,27,Factors affecting power,"Explain how sample size, variability, signific...",Unit 10: Bootstrapping and Power,2.D|4.F|4.G,0.70000,True,Includes qualitative power comparisons.
254,KC.U10.17.power_specific_alternative,27,Power for a specific alternative value,Recognize that power is evaluated for a specif...,Unit 10: Bootstrapping and Power,2.C|4.F,0.78000,True,Critical for interpreting power curves and alt...


And the observation distribution becomes :

In [ ]:
# Create a simple lookup: fine KC -> coarse KC
kc_mapping = nodes[["kc_id", "coarse_kc"]].set_index("kc_id")["coarse_kc"].to_dict()

# Map primary_kc_id to its coarse KC
obs["coarse_kc"] = obs["primary_kc_id"].map(kc_mapping)

# Count observations per coarse KC
obs_counts = obs.groupby("coarse_kc").agg(
    n_observations=("observation_id", "count"),
    n_students=("student_id", "nunique"),
).reset_index()

obs_counts['n_obs_per_stu'] = obs_counts['n_observations'] / obs_counts['n_students']


In [28]:
obs_filtered = obs[obs["primary_kc_id"].isin(valid_kcs)]
obs_filtered

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level,coarse_kc
29,S001,HW2,2,HW2_PCB_Q03,MCQ,PCB Q03,KC.U1.13.spread_iqr_range_sd,KC.U1.13.spread_iqr_range_sd|KC.U1.25.percenti...,1.00000,1,100,A,A,NaN,KC.U1.13.spread_iqr_range_sd
39,S001,HW2,2,HW2_SG1_Q04,MCQ,SG1 Q04,KC.U1.13.spread_iqr_range_sd,KC.U1.13.spread_iqr_range_sd,1.00000,1,100,A,A,NaN,KC.U1.13.spread_iqr_range_sd
40,S001,HW2,2,HW2_SG1_Q06,MCQ,SG1 Q06,KC.U1.18.linear_transformations,KC.U1.18.linear_transformations|KC.U1.13.sprea...,1.00000,1,100,C,C,NaN,KC.U1.18.linear_transformations
41,S001,HW2,2,HW2_SG1_Q34,MCQ,SG1 Q34,KC.U1.18.linear_transformations,KC.U1.18.linear_transformations|KC.U1.13.sprea...,1.00000,1,100,A,A,NaN,KC.U1.18.linear_transformations
52,S001,HW3,3,HW3_SG1_Q44,MCQ,SG1 Q44,KC.U1.18.linear_transformations,KC.U1.18.linear_transformations|KC.U1.19.z_sco...,0.00000,1,0,D,C,NaN,KC.U1.18.linear_transformations
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21713,S025,MF1,27,MF1_MCQ26,MCQ,Section I Q12,KC.U1.18.linear_transformations,KC.U1.18.linear_transformations|KC.U1.16.resis...,0.00000,1,0,B,E,NaN,KC.U1.18.linear_transformations
21716,S025,MF1,27,MF1_MCQ29,MCQ,Section I Q15,KC.U5.17.sample_mean_mean,KC.U5.17.sample_mean_mean|KC.U5.18.sample_mean...,0.00000,1,0,C,A,NaN,KC.U5.17.sample_mean_mean
21756,S025,MF2,27,MF2_MCQ09,MCQ,Section I Q09,KC.U4.01.probability_long_run_interpretation,KC.U4.01.probability_long_run_interpretation|K...,1.00000,1,100,A,A,NaN,KC.U4.01.probability_long_run_interpretation
21769,S025,MF2,27,MF2_MCQ22,MCQ,Section I Q10,KC.U1.18.linear_transformations,KC.U1.18.linear_transformations|KC.U1.13.sprea...,1.00000,1,100,B,B,NaN,KC.U1.18.linear_transformations


Testing out pyBKT with the new dataset :

In [29]:
from pyBKT.models import Model

In [53]:
# Consider any non-full marks as incorrect (replace 0.5 -> 0) 
obs['correct'] = obs['score'].case_when([
    (obs['score'] == 0.5, 0)
]).astype(int)

# Rename  columns 
bkt_data = obs.rename(columns={
    'student_id':     'user_id',
    'primary_kc_id':  'skill_name',
    'observation_id': 'order_id'
})[['user_id', 'skill_name', 'correct', 'order_id']].reset_index(drop=True)
bkt_data['order_id'] = bkt_data.groupby('user_id').cumcount()
bkt_data
bkt_data.to_csv('../data/processed/bkt_data.csv')

In [45]:
model = Model(seed=42, num_fits=1) 

model.fit(data = bkt_data)
params = model.params().reset_index()

print("FITTED BKT PARAMETERS: one row per KC")
print("=" * 65)
print(params.to_string(index=False))

FITTED BKT PARAMETERS: one row per KC
                                            skill   param   class   value
                     KC.U1.13.spread_iqr_range_sd   prior default     NaN
                     KC.U1.13.spread_iqr_range_sd  learns default 1.00000
                     KC.U1.13.spread_iqr_range_sd guesses default 0.50000
                     KC.U1.13.spread_iqr_range_sd   slips default 0.50000
                     KC.U1.13.spread_iqr_range_sd forgets default 0.00000
                  KC.U1.18.linear_transformations   prior default     NaN
                  KC.U1.18.linear_transformations  learns default 1.00000
                  KC.U1.18.linear_transformations guesses default 0.50000
                  KC.U1.18.linear_transformations   slips default 0.50000
                  KC.U1.18.linear_transformations forgets default 0.00000
          KC.U2.08.explanatory_response_variables   prior default     NaN
          KC.U2.08.explanatory_response_variables  learns default 1.00000


/Users/mailysguedon/miniforge3/envs/stellar-proj/lib/python3.11/site-packages/pyBKT/fit/M_step.py:61: RuntimeWarning: invalid value encountered in divide
  model['pi_0'] = init_softcounts[:] / np.sum(init_softcounts[:])


In [9]:
import networkx.algorithms.community as nx_comm

G_undirected = G.to_undirected()
communities = nx_comm.greedy_modularity_communities(G_undirected)

kc_to_community = {}
for i, community in enumerate(communities):
    for kc in community:
        kc_to_community[kc] = f"cluster_{i}"

nodes["coarse_kc"] = nodes["kc_id"].map(kc_to_community)

In [10]:
clusters = nodes[['kc_id','coarse_kc']].groupby('coarse_kc').agg('sum')
clusters.loc['cluster_12','kc_id']

'KC.U10.01.bootstrap_purpose_sampling_distributionKC.U10.02.bootstrap_resampling_with_replacementKC.U10.03.bootstrap_same_sample_sizeKC.U10.04.bootstrap_distribution_of_statisticKC.U10.05.bootstrap_percentile_cutoffsKC.U10.06.bootstrap_percentile_interval_constructKC.U10.07.bootstrap_interval_interpret_contextKC.U10.08.bootstrap_limitations_bias_representativenessKC.U10.09.bootstrap_vs_randomization_test'

TypeError: Series.isin() missing 1 required positional argument: 'values'